# Financial Risk Classification

In [16]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

## 1. Load Labeled Data

In [17]:
LABELED_DATA_PATH = 'archive/financial_risk_labeled.csv'
LABEL_COLUMN = 'financial_profile_label'

FEATURES_TO_KEEP = [
    'loan_to_income_ratio', 'expenses_to_income_ratio', 'savings_to_income_ratio',
    'debt_to_income_ratio', 'previous_default_count',
    'loan_duration_months', 'interest_rate', 'age', 'employment_stability_years'
]


def load_labeled_data(csv_path, seed=SEED):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f'Labeled data not found: {csv_path}. Run clustering.ipynb first.'
        )

    df = pd.read_csv(csv_path)
    required_columns = FEATURES_TO_KEEP + [LABEL_COLUMN]
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    X = df[FEATURES_TO_KEEP].values
    y = df[LABEL_COLUMN].values.astype(int)
    num_classes = len(np.unique(y))

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train_scaled, y_train)).shuffle(
        10000,
        seed=seed,
        reshuffle_each_iteration=True
    ).batch(32)
    test_dataset = tf.data.Dataset.from_tensor_slices((X_test_scaled, y_test)).batch(32)

    return train_dataset, test_dataset, scaler, FEATURES_TO_KEEP, num_classes

## 2. Custom Component (Custom Layer)

In [18]:
@tf.keras.utils.register_keras_serializable()
class CustomDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super(CustomDenseLayer, self).__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=True,
            name='custom_weight'
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='custom_bias'
        )

    def call(self, inputs):
        output = tf.matmul(inputs, self.w) + self.b
        if self.activation is not None:
            output = self.activation(output)
        return output

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                'units': self.units,
                'activation': tf.keras.activations.serialize(self.activation),
            }
        )
        return config

## 3. Build Model (Functional API)

In [19]:
def create_model(input_dim, num_classes):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(64, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = CustomDenseLayer(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(16, activation='relu')(x)

    if num_classes == 2:
        outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    else:
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='FinancialProfile_Model')
    return model

## 4. Custom Training Loop with Gradient Tape

In [20]:
def train_model(model, train_dataset, val_dataset, num_classes, epochs=30):
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    is_binary = (num_classes == 2)
    if is_binary:
        loss_fn = tf.keras.losses.BinaryCrossentropy()
        acc_metric_train = tf.keras.metrics.BinaryAccuracy()
        acc_metric_val = tf.keras.metrics.BinaryAccuracy()
    else:
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        acc_metric_train = tf.keras.metrics.SparseCategoricalAccuracy()
        acc_metric_val = tf.keras.metrics.SparseCategoricalAccuracy()

    mae_metric_train = tf.keras.metrics.MeanAbsoluteError()
    mae_metric_val = tf.keras.metrics.MeanAbsoluteError()

    @tf.function
    def train_step(x_batch, y_batch):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss_value = loss_fn(y_batch, logits)

        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        acc_metric_train.update_state(y_batch, logits)
        if is_binary:
            mae_metric_train.update_state(y_batch, logits)
        else:
            pred_classes = tf.cast(tf.argmax(logits, axis=1), tf.float32)
            y_true_f32 = tf.cast(y_batch, tf.float32)
            mae_metric_train.update_state(y_true_f32, pred_classes)

        return loss_value

    @tf.function
    def test_step(x_batch, y_batch):
        val_logits = model(x_batch, training=False)
        acc_metric_val.update_state(y_batch, val_logits)

        if is_binary:
            mae_metric_val.update_state(y_batch, val_logits)
        else:
            pred_classes = tf.cast(tf.argmax(val_logits, axis=1), tf.float32)
            y_true_f32 = tf.cast(y_batch, tf.float32)
            mae_metric_val.update_state(y_true_f32, pred_classes)

    last_metrics = {}

    for epoch in range(epochs):
        for x_batch_train, y_batch_train in train_dataset:
            train_step(x_batch_train, y_batch_train)
        for x_batch_val, y_batch_val in val_dataset:
            test_step(x_batch_val, y_batch_val)

        train_acc = float(acc_metric_train.result().numpy())
        val_acc = float(acc_metric_val.result().numpy())
        train_mae = float(mae_metric_train.result().numpy())
        val_mae = float(mae_metric_val.result().numpy())
        last_metrics = {
            'train_acc': train_acc,
            'val_acc': val_acc,
            'train_mae': train_mae,
            'val_mae': val_mae,
        }

        if epoch == epochs - 1 or epoch % 5 == 0:
            print(
                f'Epoch {epoch + 1}: '
                f'Train Acc {train_acc:.4f}, Val Acc {val_acc:.4f}, '
                f'Train MAE {train_mae:.4f}, Val MAE {val_mae:.4f}'
            )

        acc_metric_train.reset_state()
        mae_metric_train.reset_state()
        acc_metric_val.reset_state()
        mae_metric_val.reset_state()

    return model, last_metrics

## 5. Inference Script

In [21]:
def inference(model, scaler, input_dict, num_classes):
    annual_income = max(input_dict['annual_income_idr'], 1)
    loan_to_income = input_dict['loan_amount_idr'] / annual_income
    expenses_to_income = input_dict['monthly_expenses_idr'] / (annual_income / 12)
    savings_to_income = input_dict['savings_balance_idr'] / annual_income
    debt_to_income = input_dict['monthly_debt_payment_idr'] / (annual_income / 12)

    raw_input = np.array([[
        loan_to_income, expenses_to_income, savings_to_income, debt_to_income,
        input_dict['previous_default_count'],
        input_dict['loan_duration_months'], input_dict['interest_rate'],
        input_dict['age'], input_dict['employment_stability_years']
    ]])

    scaled_input = scaler.transform(raw_input)
    predictions = model.predict(scaled_input, verbose=0)[0]

    pred_class = int(np.argmax(predictions))
    prob = predictions[pred_class]

    profile_labels = {
        0: 'Financially Stable',
        1: 'Moderate Financial Capacity',
        2: 'Financially Vulnerable'
    }
    profile_label = profile_labels.get(pred_class, f'Financial Profile {pred_class}')

    print(f'Segment membership score kelas {pred_class} : {prob:.4f}')
    print(f'Profil Finansial                       : {profile_label}')
    print('Catatan: segmentasi ini bukan probabilitas gagal bayar atau keputusan kredit resmi.')

    return predictions, profile_label

## 6. Execution

In [22]:
train_dataset, test_dataset, scaler, features, num_classes = load_labeled_data(LABELED_DATA_PATH)
model = create_model(len(features), num_classes=num_classes)
trained_model, training_metrics = train_model(
    model,
    train_dataset,
    test_dataset,
    num_classes=num_classes,
    epochs=30
)
print(
    f"\nFinal test metrics: "
    f"Acc {training_metrics['val_acc']:.4f}, "
    f"MAE {training_metrics['val_mae']:.4f}"
)
trained_model.save('risk_profile_model.keras')

Epoch 1: Train Acc 0.7935, Val Acc 0.9410, Train MAE 0.2910, Val MAE 0.0752
Epoch 6: Train Acc 0.9222, Val Acc 0.9550, Train MAE 0.1133, Val MAE 0.0547
Epoch 11: Train Acc 0.9362, Val Acc 0.9750, Train MAE 0.0905, Val MAE 0.0342
Epoch 16: Train Acc 0.9473, Val Acc 0.9810, Train MAE 0.0790, Val MAE 0.0264
Epoch 21: Train Acc 0.9565, Val Acc 0.9830, Train MAE 0.0642, Val MAE 0.0225
Epoch 26: Train Acc 0.9582, Val Acc 0.9850, Train MAE 0.0620, Val MAE 0.0195
Epoch 30: Train Acc 0.9638, Val Acc 0.9910, Train MAE 0.0530, Val MAE 0.0127

Final test metrics: Acc 0.9910, MAE 0.0127


## 7. Inference Example

In [23]:
sample_input = {
    'annual_income_idr': 120_000_000,
    'loan_amount_idr': 35_000_000,
    'monthly_expenses_idr': 4_500_000,
    'savings_balance_idr': 25_000_000,
    'monthly_debt_payment_idr': 2_000_000,
    'previous_default_count': 0,
    'loan_duration_months': 24,
    'interest_rate': 8.5,
    'age': 32,
    'employment_stability_years': 6
}

predictions, profile_label = inference(trained_model, scaler, sample_input, num_classes)
pd.DataFrame({
    'financial_profile_label': list(range(num_classes)),
    'profile_name': [
        'Financially Stable',
        'Moderate Financial Capacity',
        'Financially Vulnerable'
    ][:num_classes],
    'segment_membership_score': predictions
}).sort_values('segment_membership_score', ascending=False)

Segment membership score kelas 0 : 0.9986
Profil Finansial                       : Financially Stable
Catatan: segmentasi ini bukan probabilitas gagal bayar atau keputusan kredit resmi.


,financial_profile_label,profile_name,segment_membership_score
0,0,Financially Stable,0.998551
2,2,Financially Vulnerable,0.001280
1,1,Moderate Financial Capacity,0.000169
